# Module 3a — Generate evaluation questions from your library

**Time:** ~10 min · **Output:** `workshop_outputs/03_questions.csv`, `workshop_outputs/03_answers.csv`

1. Read and chunk the same documents you uploaded to MyGPT
2. Ask an LLM to write one question + reference answer per sampled chunk
3. Add unanswerable questions
4. **Review the questions yourself**, then run them through MyGPT

In [1]:
import sys
from pathlib import Path
import re
import random

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

In [2]:
DOCS_DIR = ROOT / "data" / "BioASQ" / "pdfs_70"
N_CHUNKS = 20                              # number of questions to generate (reduce on slow laptops)
CHUNK_SIZE = 1500                          # characters per chunk used for question generation
SEED = 42

client = MyGPTClient.from_env()
print("Generator LLM:", client.llm, "via", client.llm_route())

Generator LLM: gpt-6-luna via MyGPT backend → OpenAI (http://host.docker.internal:8000)


## 1. Read and chunk documents

In [3]:
def read_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return ""

chunks = []
for path in sorted(DOCS_DIR.glob("*")):
    text = re.sub(r"\s+", " ", read_text(path)).strip()
    for start in range(0, len(text), CHUNK_SIZE):
        piece = text[start:start + CHUNK_SIZE]
        if len(piece) > 400:                      # skip tiny tail chunks
            chunks.append({"source_doc": path.name, "chunk": piece})

print(len(chunks), "chunks from", len({c["source_doc"] for c in chunks}), "documents")
random.seed(SEED)
sample = random.sample(chunks, min(N_CHUNKS, len(chunks)))

755 chunks from 70 documents


## 2. Generate question + reference answer per chunk
The prompt asks for **specific, self-contained** questions: a reader shouldn't need to see the chunk to understand the question.

In [4]:
import json

QGEN_PROMPT = """You are building a benchmark for a question-answering system over a document library.
Read the passage and write ONE question that:
- can be answered from this passage alone,
- is specific and self-contained (do not say "the passage", "this document", "the author"),
- a researcher in this field would realistically ask.
Then write a concise reference answer (1-2 sentences) using only the passage.

Return JSON: {{"question": "...", "answer": "..."}}

PASSAGE:
{chunk}
"""

rows = []
for i, item in enumerate(sample, 1):
    try:
        raw = client.ollama(QGEN_PROMPT.format(chunk=item["chunk"]), temperature=0.2, json_mode=True)
        qa = json.loads(raw)
        rows.append({"question": qa["question"].strip(), "reference_answer": qa["answer"].strip(),
                     "answerable": 1, "source_doc": item["source_doc"], "source_chunk": item["chunk"]})
        print(f"[{i}] {qa['question']}")
    except Exception as exc:
        print(f"[{i}] skipped: {exc}")

[1] Which malignancies are identified as associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2] In patients with stage II or III COPD, how did a single 12 mg dose of formoterol compare with 50 mg of salmeterol on the 0–1-hour FEV1 AUC endpoint?
[3] What factors are associated with increased susceptibility to postinfectious irritable bowel syndrome?
[4] How did Wharton's jelly stromal cell microtissues (WJSC-MT) compare in size with fibroblast microtissues (FIB-MT) during the 28-day evaluation?
[5] How does loss of PTEN affect interactions between human glioma cells and autologous T cells?
[6] Among the glioblastomas studied, how was EGFR gene amplification associated with chromosome 10 loss, and what possible sequence of events did the findings suggest?
[7] What dengue viremia level was sufficient to infect Aedes aegypti and Aedes albopictus, and what does this imply about defervescent patients?
[8] What effect does accidental mutation of a caretaker gene

## 3. Add unanswerable questions
Write questions that look plausible for your field but that your documents **do not** answer. You need these to test whether HI catches hallucinations.

In [5]:
import pandas as pd

UNANSWERABLE = [
    "What is the boiling point of tungsten?",
    "Who won the 2018 FIFA World Cup?",
    "What dose of imatinib is recommended for adults with chronic myeloid leukemia?",   # ← replace with near-miss questions for YOUR domain
]
NOT_IN_LIBRARY = "The library does not contain this information."
rows += [{"question": q, "reference_answer": NOT_IN_LIBRARY, "answerable": 0,
          "source_doc": "", "source_chunk": ""} for q in UNANSWERABLE]

questions = pd.DataFrame(rows)
questions.insert(0, "question_id", range(1, len(questions) + 1))
questions.to_csv(OUT / "03_questions.csv", index=False)
questions[["question_id", "question", "reference_answer", "answerable"]]

,question_id,question,reference_answer,answerable
0,1,Which malignancies are identified as associate...,MASLD is associated with hepatocellular carcin...,1
1,2,"In patients with stage II or III COPD, how did...",Formoterol 12 mg produced a 0.086 L greater im...,1
2,3,What factors are associated with increased sus...,Risk factors include greater pathogen virulenc...,1
3,4,How did Wharton's jelly stromal cell microtiss...,WJSC-MT were larger than FIB-MT on every day a...,1
4,5,How does loss of PTEN affect interactions betw...,"PTEN loss, associated with activation of the P...",1
5,6,"Among the glioblastomas studied, how was EGFR ...",Every glioblastoma with EGFR amplification als...,1
6,7,What dengue viremia level was sufficient to in...,Viremia above 10⁴ copies/ml could infect both ...,1
7,8,What effect does accidental mutation of a care...,It causes genetic instability that prepares th...,1
8,9,"In patient-derived glioblastoma stem cells, ho...",Treatment drives surviving cells through inter...,1
9,10,How much of the variability in sertraline and ...,Daily sertraline dose was significantly associ...,1


## 4. ✋ Human review (do not skip)
Open `workshop_outputs/03_questions.csv` in a spreadsheet or the Jupyter CSV viewer:
- **delete** questions that are vague, trivial or wrong
- **edit** reference answers that are incomplete
- check that each question really is answerable from its `source_chunk`

Save the file, then run the next cell to reload it.

In [6]:
questions = pd.read_csv(OUT / "03_questions.csv")
print(len(questions), "questions after review;", int((questions.answerable == 0).sum()), "unanswerable")

23 questions after review; 3 unanswerable


## 5. Run every question through MyGPT

In [7]:
answers = []
for i, r in questions.iterrows():
    print(f"[{i + 1}/{len(questions)}] {r.question}")
    try:
        res = client.ask(r.question)
    except Exception as exc:
        print("   failed:", exc)
        res = {"error": str(exc)}
    answers.append({**r.to_dict(), **{k: v for k, v in res.items() if k != "question"}})

answers = pd.DataFrame(answers)
answers.to_csv(OUT / "03_answers.csv", index=False)
answers[["question_id", "question", "answer", "QRS", "ARS", "HI"]].head(10)

[1/23] Which malignancies are identified as associated with metabolic dysfunction-associated steatotic liver disease (MASLD)?
[2/23] In patients with stage II or III COPD, how did a single 12 mg dose of formoterol compare with 50 mg of salmeterol on the 0–1-hour FEV1 AUC endpoint?
[3/23] What factors are associated with increased susceptibility to postinfectious irritable bowel syndrome?
[4/23] How did Wharton's jelly stromal cell microtissues (WJSC-MT) compare in size with fibroblast microtissues (FIB-MT) during the 28-day evaluation?
[5/23] How does loss of PTEN affect interactions between human glioma cells and autologous T cells?
[6/23] Among the glioblastomas studied, how was EGFR gene amplification associated with chromosome 10 loss, and what possible sequence of events did the findings suggest?
[7/23] What dengue viremia level was sufficient to infect Aedes aegypti and Aedes albopictus, and what does this imply about defervescent patients?
[8/23] What effect does accidental muta

,question_id,question,answer,QRS,ARS,HI
0,1,Which malignancies are identified as associate...,MASLD is associated with **hepatocellular carc...,61.0,90.0,27.0
1,2,"In patients with stage II or III COPD, how did...",A single 12 mg dose of formoterol was statisti...,44.0,96.0,27.0
2,3,What factors are associated with increased sus...,Factors associated with increased susceptibili...,61.0,93.0,25.0
3,4,How did Wharton's jelly stromal cell microtiss...,WJSC-MT were larger than FIB-MT at every time ...,37.0,73.0,28.0
4,5,How does loss of PTEN affect interactions betw...,PTEN loss in human glioma cells causes substan...,74.0,100.0,39.0
5,6,"Among the glioblastomas studied, how was EGFR ...",Every glioblastoma with EGFR gene amplificatio...,57.0,98.0,34.0
6,7,What dengue viremia level was sufficient to in...,Dengue viremia above **10⁴ copies/ml** was suf...,45.0,98.0,41.0
7,8,What effect does accidental mutation of a care...,"It causes genetic instability, helping the pre...",29.0,72.0,23.0
8,9,"In patient-derived glioblastoma stem cells, ho...",Cytotoxic drug treatment pushes surviving pati...,43.0,95.0,40.0
9,10,How much of the variability in sertraline and ...,Daily sertraline dose accounted for **62% of t...,43.0,98.0,33.0


➡️ Next: **03b_label_and_judge.ipynb**